# Sysmon CNN-only / LSTM-only：固定划分下的独立训练

每次只训练 `MODEL_TYPE` 指定的一个模型，分别保存结果；**不生成模型对比表、排名或自动择优功能**。

适配 Preprocess **2.0.1**。已内嵌上一轮主实验 `cnn_lstm_20261009T153018Z_561fd196` 的原始划分与配置：
512 次候选运行，355/73/84 次训练/验证/测试，输入 `[32,148]`，system 视图。
使用的原始前处理目录是 `all-families-benign-seq32-v1`。
代码核验数据指纹，不会重新生成划分；缺少匹配数据时停止并提示。

**操作**：上传本 Notebook 到 Colab，选 GPU，填写数据路径，顺序执行。
先用 `cnn_only` 跑完，再将它改为 `lstm_only` 并从配置单元向后重新运行。
辅助源码和参考划分均已内嵌，不需要另传代码或原始 XML。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. 配置

`DATA_DIR` 是 Google Drive 中与原 CNN–LSTM 实验完全相同的前处理目录。
也可设置 `DATA_ZIP`，使用完整前处理数据压缩包。
`REFERENCE_EXPERIMENT=None` 默认使用内嵌的主实验划分；其他实验可指定其结果 ZIP 或目录，必须包含 config.json、splits.csv 和 dataset_SHA256SUMS.txt。

序列长度、视图、随机种子、最大 epoch、batch size 和分组审核声明均从参考实验继承。
不要将训练权重与过采样叠加；本代码继续使用原来的加权方案。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import uuid

PROJECT = Path('/content/drive/MyDrive/SysmonResearch')
DATA_DIR = PROJECT / 'all-families-benign-seq32-v1'
DATA_ZIP = None  # 例如 PROJECT / 'all-families-benign-seq32-v1.zip'
MODEL_TYPE = 'cnn_only'  # 第二次运行改为 'lstm_only'
REFERENCE_EXPERIMENT = None  # 默认使用内嵌的 20261009 主实验；也可指定结果 ZIP / 目录
if MODEL_TYPE not in ('cnn_only', 'lstm_only'):
    raise ValueError("MODEL_TYPE 必须为 'cnn_only' 或 'lstm_only'")
OUTPUT = PROJECT / 'experiments' / (MODEL_TYPE + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8])
print('模型:', MODEL_TYPE, '\n结果目录:', OUTPUT)


## 2. 展开内嵌源码并检查环境

使用 Colab 预装 TensorFlow、NumPy、scikit-learn 和 Matplotlib，不自动升级环境。
实际依赖版本、GPU 信息及与参考实验的版本差异会保存到 config.json；固定种子不保证跨硬件版本逐位相同。
源码仅展开到临时目录，不执行网络下载。若重新跑第二个模型，从配置单元继续顺序执行即可。


In [ ]:
import base64, zlib, json, sys, tempfile, importlib, importlib.util
CODE_DIR = Path(tempfile.mkdtemp(prefix='sysmon_baselines_'))
EMBEDDED = '...'
for name, encoded in json.loads(zlib.decompress(base64.b64decode(EMBEDDED))).items():
    dest = CODE_DIR / name
    if not dest.resolve().is_relative_to(CODE_DIR.resolve()):
        raise ValueError('非法内嵌文件路径')
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(base64.b64decode(encoded))
sys.path.insert(0, str(CODE_DIR))
missing = [name for name in ('tensorflow', 'numpy', 'sklearn', 'matplotlib') if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f'缺少依赖 {missing}；安装对应依赖后重启运行时，从头执行。')
# Avoid reusing a different notebook's reader or an earlier embedded module.
for name in ('baselines', 'training_core', 'dataset', 'behavior_features', 'preprocess', 'make_splits'):
    sys.modules.pop(name, None)
importlib.invalidate_caches()
import tensorflow as tf
import baselines as B
reference = B.load_reference(REFERENCE_EXPERIMENT if REFERENCE_EXPERIMENT is not None else CODE_DIR / 'reference_cnn_lstm_main')
print('TensorFlow:', tf.__version__, 'GPU:', tf.config.list_physical_devices('GPU'))
print('参考设置:', {k: reference['config'][k] for k in ('view', 'input_shape', 'seed', 'epochs', 'batch_size')})
print('原始划分 SHA256:', reference['splits_sha256'])
print('展开源码:', CODE_DIR)


## 3. 校验数据并复用原始划分

不重新分层或随机分组，不更改前处理文件。完整读取可能需要数分钟和较多内存。
检查包含数据校验和、候选运行覆盖、group_id/SHA256 隔离、序列顺序、补齐掩码和特征形状。
若数据集不匹配，请使用原实验的数据，或明确切换到另一份实验参考；不能通过修改校验文件跳过错误。


In [ ]:
if DATA_ZIP is not None:
    DATA_DIR = B.unpack_dataset(DATA_ZIP)
B.verify_reference_dataset(DATA_DIR, reference)
data = B.load_data(DATA_DIR, view=reference['config']['view'])
indices, split_summary = B.prepare_reference_splits(data, reference)
print('X:', data['x'].shape, 'mask:', data['mask'].shape)
print(json.dumps(split_summary, ensure_ascii=False, indent=2))
if not reference['config']['software_groups_reviewed']:
    print('沿用原实验 SHA256/group 分组；产品级分组尚未人工审核，不据此宣称未见正常软件泛化。')


## 4. 训练所选模型

- **CNN-only**：清零 PAD → causal Conv1D(32,k=3/5) 并行分支 → 拼接 → 再次清零 PAD → 掩码全局平均池化。
- **LSTM-only**：清零 PAD → LSTM(64)，直接输入事件向量并显式传递掩码。
- 两者均接 Dropout(0.3) → Dense(32,ReLU) → sigmoid。

CNN-only 的池化仅平均有效位置；LSTM-only 跳过右侧补齐。
训练权重、验证权重、Adam(0.001,clipnorm=1)、早停 patience=7 与原模型一致。
参数量不强制相同，自动写入 config.json；148 维输入下 CNN-only 为 40,065，LSTM-only 为 56,641。
各模型分别在验证集上选阈值，使用相同 F2 规则，不照搬原模型的阈值数值。


In [ ]:
model, decision = B.train(data, indices, OUTPUT, MODEL_TYPE, reference)
print(json.dumps(decision, ensure_ascii=False, indent=2))
print('训练完成，模型和原始划分已保存:', OUTPUT)


## 5. 最终测试（独立执行单元）

模型、checkpoint、均值聚合和阈值已由训练/验证流程确定。
输出该模型自己的运行级与片段级指标、预测 CSV 和诊断图，不读取其他模型的分数。
同一结果目录不重复覆盖测试结果；不要根据测试成绩反复调整模型或划分。


In [ ]:
report = B.evaluate_test(model, data, indices, OUTPUT, decision)
print(json.dumps(report, ensure_ascii=False, indent=2))
fig = B.plot_results(OUTPUT)
import matplotlib.pyplot as plt
plt.show()


## 6. 重载核验与下载结果

模型重载前导入 baselines 或 training_core，以注册 ApplyMask。
只下载本次所选模型的实验结果包，不汇总跨模型比较。


In [ ]:
import numpy as np
import shutil
restored = tf.keras.models.load_model(OUTPUT / 'best.keras')
check_idx = indices['validation'][:8]
np.testing.assert_allclose(B.predict(model, data, check_idx), B.predict(restored, data, check_idx), atol=1e-6, rtol=1e-6)
print('模型重载核验通过。结果目录:', OUTPUT)
archive = shutil.make_archive(str(Path(tempfile.gettempdir()) / OUTPUT.name), 'zip', root_dir=OUTPUT)
from google.colab import files
files.download(archive)


## 手动整理结果时的说明

比较运行级 `primary_run_level`，同时保留 n、混淆矩阵、FPR/FNR 和片段级结果。
两次新实验与原实验应具有相同的 dataset_checksums_sha256、splits.csv、输入形状及主要训练设置。
这里分别去除卷积或循环模块；CNN-only 增加必要的序列池化，参数量不同，因此不是严格等参数容量比较。
原测试数据已被分析，应如实说明这是固定留出集上的补充架构实验。

掩码接口依据：[Keras GlobalAveragePooling1D](https://keras.io/api/layers/pooling_layers/global_average_pooling1d/)、[Keras LSTM](https://keras.io/api/layers/recurrent_layers/lstm/)。
